In [2]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
INTERIMDIR = CONFIGS['filepaths']['interim']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']

In [4]:
def mean(da):
    if 'time' not in da.dims:return da
    return da.mean('time',skipna=True)

In [5]:
lf,shf = [],[]
for split in ['train','valid']:
    with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{split}.h5'),engine='h5netcdf') as ds:
        lf.append(ds['lf'].load())
        shf.append(ds['shf'].load())
lf  = lf[0] if 'time' not in lf[0].dims else xr.concat(lf,dim='time')
shf = xr.concat(shf,dim='time')

lf_mean  = mean(lf)
shf_mean = mean(shf)
lf_bc    = lf.broadcast_like(shf) if 'time' not in lf.dims else lf
max_mean = mean(xr.apply_ufunc(np.maximum,lf_bc,shf))

with xr.open_dataset(os.path.join(INTERIMDIR,'sst.nc')) as ds:
    sst = ds['sst'].load()
sst_mean = mean(sst)
sst_anom = sst_mean - sst_mean.mean(skipna=True)

In [ ]:
clim    = float(max(np.nanmax(np.abs(lf_mean.values)),np.nanmax(np.abs(shf_mean.values)),np.nanmax(np.abs(max_mean.values))))
sst_lim = float(np.nanmax(np.abs(sst_anom.values)))

kw  = dict(cmap='ColdHot',vmin=-clim,vmax=clim,levels=21)
fmt = dict(coast=True,borders=False,latlim=LATRANGE,lonlim=LONRANGE,latlines=[10,15,20],lonlines=5,grid=False)

fig,axs = pplt.subplots(nrows=2,ncols=2,proj='cyl',figwidth=6.5,share=True)
axs.format(**fmt)
axs[-1,:].format(lonlabels='b')
axs[:,0].format(latlabels='l')

m     = axs[0].pcolormesh(lf_mean.lon,lf_mean.lat,lf_mean,**kw)
axs[1].pcolormesh(shf_mean.lon,shf_mean.lat,shf_mean,**kw)
axs[2].pcolormesh(max_mean.lon,max_mean.lat,max_mean,**kw)
m_sst = axs[3].pcolormesh(sst_anom.lon,sst_anom.lat,sst_anom,cmap='ColdHot',vmin=-sst_lim,vmax=sst_lim,levels=21)

axs[0].format(title=r'$\widetilde{LF}$')
axs[1].format(title=r'$\widetilde{SHF}$')
axs[2].format(title=r'$\max(\widetilde{LF},\widetilde{SHF})$')
axs[3].format(title='SST Anomalies')

fig.colorbar(m,ax=[axs[0],axs[2]],loc='b',label='z-score')
fig.colorbar(m_sst,ax=axs[3],loc='b',label='K')
pplt.show()
fig.save('../figs/fig_2.jpg')